In [ ]:
"""
Eksport bazy lokalizacji ze scoringiem do XLSX - to jest wymagane wprost
w briefie ("baza lokalizacji z punktacja i szacunkiem popytu, XLSX").

Cztery arkusze: top 800 korytarzowe, top 800 docelowe (oba z uzasadnieniem),
wszystkie lokalizacje (kluczowe kolumny, nie wszystkie ~100 bo nikt by tego
nie przeczytal), i legenda co znaczy kazda kolumna.

Wymaga: candidate_locations_ze_scoringiem.csv, top800_lokalizacji.xlsx
Wynik: baza_lokalizacji_ev.xlsx
"""

import os
import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule

PLIK_SCORING = "../data/candidate_locations_ze_scoringiem.csv" if os.path.exists("../data/candidate_locations_ze_scoringiem.csv") else "candidate_locations_ze_scoringiem.csv"
PLIK_TOP800 = "../data/top800_lokalizacji.xlsx" if os.path.exists("../data/top800_lokalizacji.xlsx") else "top800_lokalizacji.xlsx"
PLIK_WYJSCIOWY = "../data/baza_lokalizacji_ev.xlsx" if os.path.exists("../data") else "baza_lokalizacji_ev.xlsx"

CZCIONKA = "Arial"
KOLOR_NAGLOWKA = "1A1A2E"

df = pd.read_csv(PLIK_SCORING, low_memory=False)
top20_kor = pd.read_excel(PLIK_TOP800, sheet_name="Korytarzowa")
top20_dest = pd.read_excel(PLIK_TOP800, sheet_name="Docelowa")

# arkusz "wszystkie lokalizacje" - tylko kluczowe kolumny, zeby dalo sie
# to w ogole przegladac
KOLUMNY_GLOWNE = [
    "location_id", "name", "source_layer", "segment", "powiat_nazwa",
    "latitude", "longitude",
    "wynik_scoringowy", "ranking_scoringowy_procentyl",
    "sesje_rocznie_szacunek", "energia_kwh_rocznie_szacunek",
    "skladowa_udzial_konkurencji", "skladowa_pewnosc_danych",
    "skladowa_luka_afir", "skladowa_luka_infrastrukturalna",
    "skladowa_sklonnosc_publiczna",
    "dane_prawdopodobnie_zanizone",
    "traffic_primary_sam_osobowe", "existing_eipa_power_kw_active_2km",
    "dedup_status",
]
dostepne_kolumny = [k for k in KOLUMNY_GLOWNE if k in df.columns]

if "dedup_status" in df.columns:
    wszystkie = df[df["dedup_status"] == "unique"][dostepne_kolumny].copy()
else:
    wszystkie = df[dostepne_kolumny].copy()

if "wynik_scoringowy" in wszystkie.columns:
    wszystkie = wszystkie.sort_values("wynik_scoringowy", ascending=False)

LEGENDA = pd.DataFrame([
    ("location_id", "Unikalny identyfikator lokalizacji w tym projekcie"),
    ("name", "Nazwa z OSM/EIPA (jesli dostepna)"),
    ("source_layer", "Typ: fuel_station/junction/mop = kandydat pod nowa stacje; eipa_station/osm_charging_station = juz istnieje"),
    ("segment", "korytarzowa (MOP/wezly/TEN-T) lub docelowa (miasto/podmiejskie)"),
    ("powiat_nazwa", "Powiat, w ktorym lezy lokalizacja"),
    ("latitude, longitude", "Wspolrzedne geograficzne (WGS84)"),
    ("wynik_scoringowy", "Glowny wynik: szacowana energia/rok (kWh), skorygowana o pewnosc danych i luke AFIR"),
    ("ranking_scoringowy_procentyl", "Pozycja w rankingu SWOJEGO segmentu, 0-1 (1 = najlepsza)"),
    ("sesje_rocznie_szacunek", "Szacowana liczba sesji ladowania rocznie"),
    ("energia_kwh_rocznie_szacunek", "Surowy szacunek popytu PRZED korekta scoringowa"),
    ("skladowa_udzial_konkurencji", "Udzial lokalizacji w modelu grawitacyjnym (mniej konkurencji = wyzej)"),
    ("skladowa_pewnosc_danych", "Mnoznik pewnosci dopasowania ruchu (tylko korytarzowa): 1.0=wysoka, 0.9=srednia, 0.75=niska"),
    ("skladowa_luka_afir", "Premia za wypelnianie luki w wymaganym rozstawie hubow AFIR (tylko TEN-T, do +20%)"),
    ("skladowa_luka_infrastrukturalna", "Mnoznik za (nie)doinwestowanie powiatu wzgledem floty EV (tylko docelowa)"),
    ("skladowa_sklonnosc_publiczna", "Skłonnosc do ladowania publicznego w powiecie (tylko docelowa)"),
    ("dane_prawdopodobnie_zanizone", "True = powiat nalezy do 45 scalonych obszarow, dane EV prawdopodobnie zanizone"),
    ("traffic_primary_sam_osobowe", "Ruch drogowy (SDRR, sam. osobowe/dobe) z GPR GDDKiA"),
    ("existing_eipa_power_kw_active_2km", "Laczna moc (kW) aktywnej konkurencji w promieniu 2 km"),
    ("dedup_status", "unique = lokalizacja liczona raz; duplicate_of_eipa = ten sam punkt widoczny tez w EIPA"),
], columns=["Kolumna", "Opis"])


def formatuj_arkusz(ws, df_zrodlowe, szerokosc_pierwszej_kolumny=None):
    """naglowek, czcionka, zamrozony wiersz, dopasowane szerokosci kolumn"""
    for kom in ws[1]:
        kom.font = Font(name=CZCIONKA, bold=True, color="FFFFFF", size=11)
        kom.fill = PatternFill("solid", fgColor=KOLOR_NAGLOWKA)
        kom.alignment = Alignment(vertical="center", wrap_text=True)
    for wiersz in ws.iter_rows(min_row=2):
        for kom in wiersz:
            kom.font = Font(name=CZCIONKA, size=10)
    ws.freeze_panes = "A2"
    ws.row_dimensions[1].height = 30
    for i, kolumna in enumerate(df_zrodlowe.columns, start=1):
        litera = get_column_letter(i)
        maks_dlugosc = max(len(str(kolumna)), df_zrodlowe[kolumna].astype(str).str.len().max() if len(df_zrodlowe) else 10)
        ws.column_dimensions[litera].width = min(max(maks_dlugosc + 2, 10), 50)
    if szerokosc_pierwszej_kolumny:
        ws.column_dimensions["A"].width = szerokosc_pierwszej_kolumny


with pd.ExcelWriter(PLIK_WYJSCIOWY, engine="openpyxl") as writer:
    top20_kor.to_excel(writer, sheet_name="Top 800 - korytarzowa", index=False)
    top20_dest.to_excel(writer, sheet_name="Top 800 - docelowa", index=False)
    wszystkie.to_excel(writer, sheet_name="Wszystkie lokalizacje", index=False)
    LEGENDA.to_excel(writer, sheet_name="Legenda kolumn", index=False)

    formatuj_arkusz(writer.sheets["Top 800 - korytarzowa"], top20_kor)
    if "uzasadnienie" in top20_kor.columns:
        idx = top20_kor.columns.get_loc("uzasadnienie") + 1
        writer.sheets["Top 800 - korytarzowa"].column_dimensions[get_column_letter(idx)].width = 70

    formatuj_arkusz(writer.sheets["Top 800 - docelowa"], top20_dest)
    if "uzasadnienie" in top20_dest.columns:
        idx = top20_dest.columns.get_loc("uzasadnienie") + 1
        writer.sheets["Top 800 - docelowa"].column_dimensions[get_column_letter(idx)].width = 70

    formatuj_arkusz(writer.sheets["Wszystkie lokalizacje"], wszystkie)

    formatuj_arkusz(writer.sheets["Legenda kolumn"], LEGENDA, szerokosc_pierwszej_kolumny=35)
    writer.sheets["Legenda kolumn"].column_dimensions["B"].width = 90

    # skala kolorow na wyniku - zeby od razu bylo widac gdzie sa najlepsze
    # miejsca, nawet przy przewijaniu tysiecy wierszy
    ws_wszystkie = writer.sheets["Wszystkie lokalizacje"]
    if "wynik_scoringowy" in wszystkie.columns:
        kolumna_wyniku = wszystkie.columns.get_loc("wynik_scoringowy") + 1
        litera_wyniku = get_column_letter(kolumna_wyniku)
        zakres = f"{litera_wyniku}2:{litera_wyniku}{len(wszystkie)+1}"
        regula = ColorScaleRule(
            start_type="min", start_color="F8696B",
            mid_type="percentile", mid_value=50, mid_color="FFEB84",
            end_type="max", end_color="63BE7B",
        )
        ws_wszystkie.conditional_formatting.add(zakres, regula)

print(f"Zapisano: {PLIK_WYJSCIOWY}")
print(f"  Arkusz 'Top 800 - korytarzowa': {len(top20_kor)} wierszy")
print(f"  Arkusz 'Top 800 - docelowa': {len(top20_dest)} wierszy")
print(f"  Arkusz 'Wszystkie lokalizacje': {len(wszystkie)} wierszy, {len(dostepne_kolumny)} kolumn")
print(f"  Arkusz 'Legenda kolumn': {len(LEGENDA)} pozycji")
